# The Fleet App — `amygda_ops_risk_score`

## What this notebook does

Continues from `03_deploy_and_monitor_guide.ipynb`. That guide ends with a monitor that
scores new files and writes one fixed dashboard. Here the monitor becomes a small **app on
your own machine**:

- **you choose what it shows**: the tabs, the panels on each, the figures at the top, the
  colours and your logo, all from one small settings file (the *app spec*);
- it has buttons to **add files**, **run now**, and **set, pause or resume a schedule**;
- it raises **alerts** when something you care about happens;
- it keeps its data in ordinary **files** or in one **database file**, as you prefer.

Nothing here needs a server or any extra install. The app listens on this machine only.

| Section | What you do |
|---|---|
| 1 | Create a monitor and score the first weeks |
| 2 | See what the app can show |
| 3 | Write an app spec, and what happens when it has a mistake |
| 4 | Apply it and look at the result |
| 5 | Colours and a logo |
| 6 | Alerts |
| 7 | Start the app: add a file, run it |
| 8 | Schedule runs, pause and resume |
| 9 | Keep the app running |
| 10 | Files or a database |
| 11 | Let the agent skill do all of this by asking you questions |
| 12 | Clean up |

It uses the sample data: a fixed-log model trained on 2023, and the logs of Jan–Mar 2024
split into 13 weekly files (`sample_data/batches/`). A compact version is
`04a_fleet_app_fixed_log.ipynb`.

### Every time you open this notebook — run these cells first

In [ ]:
import glob, json, os, shutil, sqlite3, subprocess, sys, time, urllib.request, webbrowser
from pathlib import Path

import pandas as pd

# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

import amygda_ops_risk_score
from amygda_ops_risk_score import OpsRiskClient, helpers
from amygda_ops_risk_score.monitor import Monitor, SpecError, validate_spec

# The `amygda-risk` command. If this notebook's kernel can't find it on PATH (its
# environment isn't activated), the same command is run through this kernel's Python.
AMYGDA_RISK = shutil.which("amygda-risk") or f'"{sys.executable}" -m amygda_ops_risk_score.monitor.cli'

print("amygda_ops_risk_score version:", amygda_ops_risk_score.__version__)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

API_KEY = os.environ.get("AMYGDA_SDK_API_KEY") or "paste_your_api_key_here"   # ← paste your key here

# The monitor, the `amygda-risk` command and the app read the key from AMYGDA_API_KEY.
# It is never written to the monitor folder.
os.environ["AMYGDA_API_KEY"] = API_KEY

client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
# The complete model zip from Step 12: yours from examples 02a/02b if you ran them,
# otherwise the ready-made sample model (fixed-log, trained on the 2023 sample data).
MODEL_ZIPS = sorted(glob.glob("../examples/outputs/fixed_log/complete_model_*.zip"), key=os.path.getmtime)
MODEL      = MODEL_ZIPS[-1] if MODEL_ZIPS else "../examples/sample_data/models/fixed_log_complete_model.zip"

SAMPLES    = Path("../examples/sample_data")
TRAINING   = SAMPLES / "fixed_log_training.csv"        # the logs the model was trained on
BATCHES    = SAMPLES / "batches" / "fixed_log"         # week_01.csv … week_13.csv (Jan–Mar 2024)

MON_DIR    = Path("outputs/guide_fleet_app")           # the monitor folder
MON_NAME   = "guide-fleet-app"                         # deployment name, unique in your organisation
SPEC_FILE  = Path("outputs/guide_fleet_app_spec.json") # the app spec you will write
PORT       = 8050                                      # the app's address is http://127.0.0.1:8050/

week = {int(p.stem[-2:]): p for p in sorted(BATCHES.glob("week_*.csv"))}
print("Model:", MODEL)
print(f"{len(week)} weekly files in {BATCHES}")

In [ ]:
# Start clean: remove what an earlier run of this notebook left behind
for d in client.list_deployments(API_KEY):
    if d["name"] == MON_NAME:
        client.delete_deployment(API_KEY, d["deployment_id"])
        print("Deleted earlier deployment", d["name"])
shutil.rmtree(MON_DIR, ignore_errors=True)

---
## 1. Create a monitor and score the first weeks

The app sits on top of a monitor folder (section 6 of the deploy and monitor guide). This
deploys the model, creates the folder, seeds it with the training logs as history, and
scores the first four weekly files so this guide has something to show. (Creating a
monitor scores nothing by itself: in everyday use the app starts empty and you add your
files on the page.)

`storage_kind` chooses where the scores, run records and alerts are kept: `"files"` (the
default) or `"sqlite"` (one database file). Section 10 shows the difference and how to
switch later.

In [ ]:
mon = Monitor.init(
    MON_DIR,
    model          = MODEL,
    name           = MON_NAME,
    api_key        = API_KEY,
    fleet_name     = "Baggage sortation stations",
    history        = TRAINING,
    storage_kind   = "files",
)
for n in range(1, 5):
    r = mon.run(week[n])
    print(f"week {n}: {r.record['rows_new']} new logs, {r.record['periods_upserted']} periods scored")
print("Alert levels:", {k: mon.config["bands"][k] for k in ("watch", "high")})

---
## 2. See what the app can show

Everything on the page comes from a **catalogue** that ships with the SDK: *panels* (a
table or chart), *header tiles* (one figure each) and a few *settings*. `amygda-risk panels`
prints it. A newer SDK version can add panels, so look here instead of relying on a list
in a document.

**What you should see:** the panels with one line each (and the settings a panel takes,
indented), then the tiles, the settings and the rules.

In [ ]:
!{AMYGDA_RISK} panels

**Reading it:**
- Panels marked `[after the first run]` need a monitor run before they have anything to
  show; `[if the model has it]` depends on the model (for example, log codes the model
  doesn't use exist only for fixed-log models).
- The three `asset_*` panels share one asset selector, so they go on the same tab.
- `amygda-risk panels --json` (or `catalogue()` in Python) gives the same as data.

---
## 3. Write an app spec

The spec is a small JSON file. It says which tabs there are, which panels go on each (top
to bottom), which figures sit at the top, and how assets are ranked. It holds no code.
Everything is optional: what you leave out keeps its default.

This one is for someone who checks the fleet each morning: what changed first, then which
system is behind each asset's risk, then the ten worst assets.

In [ ]:
spec = {
    "title": "Sortation stations",
    "rank_by": "peak",                     # "worst" means the highest score reached
    "header": {"tiles": ["open_alerts", "assets_high_now", "highest", "last_run"]},
    "tabs": [
        {"title": "Fleet",  "panels": ["since_last_run", "fleet_systems",
                                        {"type": "ranking", "top": 10}]},
        {"title": "Alerts", "panels": ["alerts"]},
        {"title": "Assets", "panels": ["asset_trend", "asset_systems", "asset_drivers"]},
        {"title": "Health", "panels": ["run_health", "run_table", "known_gaps"]},
    ],
}
checked = validate_spec(spec)              # raises SpecError if something is wrong
print("Tabs:", [(t["title"], [p["type"] for p in t["panels"]]) for t in checked["tabs"]])

**A panel is a name, or an object when it has settings**: `"ranking"` shows every asset;
`{"type": "ranking", "top": 10}` shows the first ten. An object can also carry a `"title"`
(to rename the panel) and `"width": "half"` (two half-width panels sit side by side).

**Mistakes are explained in plain words.** A misspelt name gets a suggestion:

In [ ]:
for bad in ({"tabs": [{"title": "Fleet", "panels": ["heatmpa"]}]},
            {"tabs": [{"title": "A", "panels": ["asset_trend"]},
                      {"title": "B", "panels": ["asset_drivers"]}]},
            {"header": {"tiles": ["open_alerts", "higest"]}}):
    try:
        validate_spec(bad)
    except SpecError as e:
        print("✗", e)

---
## 4. Apply it and look at the result

`amygda-risk design <folder> --spec <file>` checks the spec, saves it in the monitor folder
as `app.json` and rebuilds the page. `--reset` goes back to the standard layout. From
Python the same is `mon.set_spec(spec)` then `mon.dashboard()`.

**What you should see:** the page opens with your four tabs. The Fleet tab starts with
"Since the last run".

In [ ]:
SPEC_FILE.write_text(json.dumps(spec, indent=1))
!{AMYGDA_RISK} design {MON_DIR} --spec {SPEC_FILE}

dash = (MON_DIR / "dashboard" / "index.html").resolve()
_ = webbrowser.open(dash.as_uri())

This page is a plain file: it shows the layout but has no buttons. The app (section 7)
shows the same layout live, with the controls added.

**Before anything is deployed or scored** you can preview a layout with the scores saved in
the model zip. Nothing is sent anywhere, and `--open` opens the page in your browser:

```bash
amygda-risk design --spec my_app.json --model complete_model_<id>.zip --open
```

Panels that need a run appear as "pending" cards, so you can see where they will go. The
same works for a monitor that has no runs yet:
`amygda-risk design <folder> --model complete_model_<id>.zip --open`.

**The title** in the spec is shown at the top exactly as you write it. Without one, the
page is called "<fleet name> risk".

---
## 5. Colours and a logo

`theme` sets the look:

| Setting | Values |
|---|---|
| `mode` | `"auto"` (follow each viewer's device, the default), `"light"` or `"dark"` |
| `accent` | Your brand colour as `"#rrggbb"`: the bar at the top and the open tab |
| `logo` | An image file (png, jpg, svg, gif or webp, up to 300 KB), shown above the title. The path is relative to the spec file; the image is copied into the monitor folder |

Amber and red can't be changed: on every chart they mean *watch* and *high*.

In [ ]:
spec["theme"] = {"mode": "light", "accent": "#0b6e4f"}
# spec["theme"]["logo"] = "my_logo.png"      # ← a logo file next to the spec file, if you have one

SPEC_FILE.write_text(json.dumps(spec, indent=1))
!{AMYGDA_RISK} design {MON_DIR} --spec {SPEC_FILE}
_ = webbrowser.open(dash.as_uri())

---
## 6. Alerts

Alert **rules** are checked after every run. Each has a type and one setting:

| Rule | Raised when | Setting (default) |
|---|---|---|
| `enters_band` | An asset reaches watch or high in a run, and wasn't already there | `band`: `"high"` (or `"watch"`, which also covers high) |
| `rises_by` | An asset's latest score rose by at least N points in one run | `points`: 10 |
| `no_data` | An asset has had no logs for N days while others have | `days`: 7 |
| `run_failed` | N runs in a row failed | `runs`: 1 |
| `unmatched_above` | More than N% of a run's new logs matched no subsystem | `percent`: 30 |

With no `alerts` in the spec, two rules apply: an asset reaching high, and a failed run.
`"rules": []` turns alerts off.

**Choose rules that can fire.** In the sample data no asset reaches the watch level, so a
band rule alone would leave the Alerts tab empty, which could be mistaken for a healthy
fleet. Check your levels against your scores, and add a rise rule sized to your data:

In [ ]:
scores, bands = mon.scores, mon.config["bands"]
print(f"watch {bands['watch']}, high {bands['high']};  highest score so far: "
      f"{scores['operational_risk'].max():.1f}")

# How much each asset's latest score moved in each run so far
moves = pd.DataFrame([c for r in mon.runs() for c in r.get("changes", [])])["change"].dropna()
print("Largest rises per run:", sorted(moves.round(1), reverse=True)[:8])

In [ ]:
spec["alerts"] = {"rules": [
    {"type": "enters_band", "band": "watch"},
    {"type": "rises_by", "points": 5},
    {"type": "no_data", "days": 7},
    {"type": "run_failed", "runs": 1},
]}
SPEC_FILE.write_text(json.dumps(spec, indent=1))
!{AMYGDA_RISK} design {MON_DIR} --spec {SPEC_FILE}

for n in range(5, 8):                      # three more weeks, with the rules in place
    r = mon.run(week[n])
    print(f"week {n}: alerts raised: {r.record.get('alerts_raised', [])}")

In [ ]:
!{AMYGDA_RISK} alerts {MON_DIR}

**How an alert lives:**
- It is **open** until someone **acknowledges** it (a button in the app, or
  `amygda-risk alerts <folder> --ack <id>`).
- While it is open, the same thing happening again **updates** it (the message, and a count
  of runs) instead of adding another.
- Alerts about a state (a failed run, a quiet asset, unmatched logs) are **cleared** by
  themselves when a later run shows the problem has ended.
- Alerts show in the app and on the command line. They are not emailed or sent anywhere.

In [ ]:
open_now = mon.alerts("open")
if open_now:
    mon.acknowledge_alerts(open_now[0]["id"])          # the same as the button in the app
pd.DataFrame(mon.alerts())[["id", "rule", "asset_id", "severity", "status", "times", "message"]]

---
## 7. Start the app

```bash
amygda-risk app outputs/guide_fleet_app
```

starts the app and opens it in your browser. It keeps running until you stop it (Ctrl+C in
that terminal). Here the notebook starts it in the background so the next cells can use it.

**What you should see:** the page from section 4, with a bar at the top: a status line (the
last run, the schedule), **Add files**, **Run now** and **Schedule**.

In [ ]:
app = subprocess.Popen([sys.executable, "-m", "amygda_ops_risk_score.monitor.cli",
                        "app", str(MON_DIR), "--port", str(PORT), "--no-open"])
time.sleep(3)
_ = webbrowser.open(f"http://127.0.0.1:{PORT}/")

In [ ]:
APP_URL = f"http://127.0.0.1:{PORT}"

def app_call(method, path, body=None):
    """Call the app the way its page does."""
    data = body if isinstance(body, bytes) else (json.dumps(body).encode() if body is not None else None)
    req = urllib.request.Request(APP_URL + path, data=data, method=method,
                                 headers={"X-Amygda-App": "1"})
    with urllib.request.urlopen(req, timeout=60) as res:
        return json.loads(res.read())

def wait_for_run():
    """Wait until the app has no run in progress, then return its state."""
    while True:
        state = app_call("GET", "/api/state")
        if not state["running"]:
            return state
        time.sleep(1)

state = app_call("GET", "/api/state")
{k: state[k] for k in ["fleet_name", "key_present", "schedule_text", "open_alerts", "last_run"]}

**Add a file and run it.** On the page, press **Add files** (or drag a file onto the page)
and then **Run now**. The files wait in the monitor's `inbox/` folder until the run; you can
also copy files there yourself. The cell below does exactly what the two buttons do.

**What you should see:** the page reloads by itself when the run finishes, with week 8 in
"Since the last run".

In [ ]:
app_call("PUT", f"/api/inbox/{week[8].name}", week[8].read_bytes())   # Add files
print("Waiting:", [f["name"] for f in app_call("GET", "/api/state")["inbox"]])

app_call("POST", "/api/run")                                          # Run now
state = wait_for_run()
print("Last run:", state["last_run"]["status"], "·", state["last_run"]["rows_new"], "new logs")

- If several files are waiting, one run scores them all together, oldest first, and moves
  them to `inbox/processed/`.
- A run that fails changes nothing: the file stays in the inbox, the page shows the reason,
  and (with the `run_failed` rule) an alert is raised that clears on the next good run.
- The app only answers requests from this machine, and only its own page can change
  anything.

---
## 8. Schedule runs, pause and resume

Press **Schedule** on the page to choose how often the app checks the inbox: every N hours,
every day at a time, or every week on a day. **Pause** and **Resume** are in the same place.
The schedule is kept in the monitor's `app.json`, so it can also be part of the spec:

```json
"schedule": {"every": "day", "at": "06:00"}
"schedule": {"every": "hours", "hours": 6}
"schedule": {"every": "week", "weekday": "mon", "at": "06:00"}
```

Times are local. A scheduled run scores whatever is waiting in the inbox; when nothing is
waiting it does nothing and records nothing. The cell below does what the Schedule form
does.

In [ ]:
app_call("POST", "/api/schedule", {"every": "day", "at": "06:00"})
state = app_call("GET", "/api/state")
print(state["schedule_text"], "· next run:", state["next_run"])

app_call("POST", "/api/schedule/pause")
print(app_call("GET", "/api/state")["schedule_text"])
app_call("POST", "/api/schedule/resume")
print(app_call("GET", "/api/state")["schedule_text"])

- Applying a new layout (`amygda-risk design`) keeps a schedule set in the app, unless the
  new spec names one.
- **The schedule only fires while the app is running.** A check missed while it was off is
  made once when it starts again. The next section keeps it running.

In [ ]:
app.terminate(); app.wait()      # stop the app this notebook started
print("App stopped.")

---
## 9. Keep the app running

`amygda-risk service <folder>` registers the app with your system, so it starts when you
log in and is restarted if it stops:

| System | What is installed |
|---|---|
| macOS | A launchd agent in `~/Library/LaunchAgents` |
| Linux | A systemd user service in `~/.config/systemd/user` |
| Windows | A Task Scheduler task that starts at log-on |

`--name` gives the entry a name of your own (letters and digits); without it the name is
made from the folder. The name and port are remembered, so later commands need neither.

Without `--install` it only **shows** what it would install, and changes nothing:

In [ ]:
!{AMYGDA_RISK} service {MON_DIR} --port {PORT} --name guide-fleet

To install it, and later to remove it:

```bash
amygda-risk service outputs/guide_fleet_app --name guide-fleet --install
amygda-risk service outputs/guide_fleet_app --remove
```

**The API key.** The system starts the app with no terminal to set the key in, so the app
reads it from a file named `.env` in the monitor folder or the folder above it:

```
AMYGDA_API_KEY=your_key_here
```

Keep that file readable only by you. If your `.env` uses another name, create the monitor
with `api_key_env="THAT_NAME"` (`--api-key-env` on the command line). The key is never
written into the monitor folder's own files or the system entry.

The machine must be on and you must be logged in for scheduled runs. The app's output goes
to `app.log` in the monitor folder.

---
## 10. Files or a database

A monitor keeps its scores, run records and alerts in one of two ways. The results are
identical; pick by how you want to use the data.

| Storage | Where | Pick it when |
|---|---|---|
| `files` (default) | `scores/scores.parquet`, `runs/<run_id>/run.json`, `alerts/alerts.json` | You read scores with pandas, or don't mind either way |
| `sqlite` | One file, `monitor.db` | You want to query scores with SQL or a reporting tool, or back up one file |

Choose at the start (`storage_kind="sqlite"`, or `amygda-risk init --storage sqlite`), or
move an existing monitor either way. Everything is copied and checked before the old copy
is removed. Stop the app first.

In [ ]:
before = mon.scores
!{AMYGDA_RISK} storage {MON_DIR} --to sqlite

mon = Monitor(MON_DIR)                     # open it again: it now reads monitor.db
pd.testing.assert_frame_equal(mon.scores, before)
print("Same scores:", len(mon.scores), "rows.  In the folder:", sorted(p.name for p in MON_DIR.iterdir()))

The database has plain tables: `scores` (one row per asset and period, with
`operational_risk` and a column per system and subsystem score), `runs` and `alerts`. Any
SQLite tool can read it. Read only: the monitor rewrites the scores on every run.

In [ ]:
con = sqlite3.connect(MON_DIR / "monitor.db")
top = pd.read_sql_query(
    """SELECT asset_id, ROUND(MAX(operational_risk), 1) AS highest, COUNT(*) AS periods
       FROM scores GROUP BY asset_id ORDER BY highest DESC LIMIT 5""", con)
runs = pd.read_sql_query("SELECT run_id, status, trigger, started_at FROM runs ORDER BY run_id", con)
con.close()
display(top, runs)

The raw-log history, the scoring bundle, the inbox and `app.json` are files either way.
`amygda-risk storage <folder>` says which kind a monitor uses.

---
## 11. Let the agent skill do all of this

Everything above can be done by answering questions instead. With the SDK's agent skill
installed (`amygda-risk-skill install`), ask your coding agent for a fleet monitoring app.
It asks three short rounds of questions:

1. **The name, and what you need to see**: what the app is called, which panels you want
   and how they are grouped into tabs, then who will use it most, which decides how they
   are laid out. It then proposes a layout for you to correct.
2. **How it looks**: light or dark, a brand colour, a logo.
3. **Alerts, schedule and storage**: what should raise an alert, how often to check for new
   files, whether the app should start by itself when you log in, and files or a database.

It writes the spec from your answers, shows you the page using the model's own scores,
changes it until you are happy, deploys the model and starts the app. It scores nothing
while setting up: the app opens waiting for your data. It never asks about ports or
servers, and it never writes app code: the app is the one in this guide, so it is the same
tested app for everyone and improves when you upgrade the SDK.

---
## 12. Clean up

`amygda-risk remove <folder>` takes a monitor away in one go: it stops the app, removes the
keep-running entry if there is one, deletes the deployment from the API and deletes the
folder. Without `--yes` it only lists what it would do. Skip this cell to keep the monitor.

In [ ]:
!{AMYGDA_RISK} remove {MON_DIR} --yes
SPEC_FILE.unlink(missing_ok=True)